# 🌇 **A Health-Relevant Computational Framework for Just Resilience: Linking Targeted Urban Cooling Scenarios to Population-Wide Benefits**

---

### **Context**

Mesoscale modeling is widely used to assess the potential of cool roofs to mitigate urban overheating and contribute to adaptation by reducing both outdoor and indoor heat stress. The prevailing approach, though, applies them uniformly across a city, offering limited guidance on practical deployment, local needs, and existing disparities. Spatially and socioeconomically focused assessment frameworks respond to this gap. This notebook reproduces such a targeted approach to the **Athens Urban Area (AUA)**, a recognized Mediterranean climate change hotspot, during the extreme **nine-day heat wave of 28 July to 5 August 2021** in Greece, using the Weather Research and Forecasting Model (WRF v4.3.2) coupled with the BEP/BEM multi-layer urban canopy scheme. Within this modeling framework, cool roofs are applied across the Elaionas zone, an industrial area of flat-roof buildings, targeted especially to benefit West Athens, a low-income AUA regional unit with limited green cooling services ([Giannaros et al., 2026](https://doi.org/10.1088/2752-5309/ae6095)). In three stages, the workflow quantifies the atmospheric cooling produced by the applied cool-roof scenarios, characterizes the heat-stress response through the **modified Physiologically Equivalent Temperature (mPET)**, and links outcomes to health-beneficial implications, emphasizing **just resilience**.

> **Note:** The accompanying slides in docs/ detail and document the study background and the methodological framework implemented below.

### **Learning Goals:**

🎯 **Goal 1 | Atmospheric Cooling**: Map the daytime air-temperature difference produced by each cool-roof scenario under varying wind patterns to compare cooling performance across the intervention's broader influence zone.

🎯 **Goal 2 | Heat-Stress Response (mPET)**: Compare a fixed heat-stress limit against a location- and group-specific adjusted limit to map where prolonged heat stress persists for different population groups.

🎯 **Goal 3 | Health-Related Benefits & Just Resilience**: Link outcomes to health-relevant implications, estimate the benefited area and population across different population groups, and discuss the relevant outcomes through the lens of distributional, recognitional, and procedural justice.

---

**Run the notebook via Binder:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/coolscape-icb2026/HEAD?labpath=notebooks%2Fcoolscape_icb2026.ipynb)

## ⚙️ Setup and Configuration

First, let's import the necessary libraries and configure our environment.


In [ ]:
# Import required libraries
import sys
import warnings
from pathlib import Path

# Core data science stack
import numpy as np
import pandas as pd
import xarray as xr

# Visualization
import matplotlib.pyplot as plt

# Geospatial
import cartopy.crs as ccrs

# Interactive mapping and widgets
import folium
import branca.colormap as bcm
import ipywidgets as widgets
from IPython.display import display, HTML

# Workshop utilities
sys.path.insert(0, str(Path('..').resolve()))

from utils.configuration import (
    SCENARIOS, REGIONS, REGION_COLORS, POPULATION_GROUPS,
    MPET_CLASSES, MPET_COLORS, MAP_EXTENT,
)
from utils.wrf_fields import load_base_case_area_means, phase_spans
from utils.mpet_data import (
    load_strong_heat_stress_thresholds, lookup_daily_strong_heat_stress_thresholds,
    load_mpet_group, load_rayman_output,
    hourly_category_frequencies, count_prolonged_exposure_days,
)
from utils.population import (
    read_gpkg_pop_weights, regional_pop_total,
    identify_benefited_points, weight_to_population,
)
from utils.visualization import (
    plot_base_t2_timeseries, plot_diff_panel_maps, plot_cooling_heatmap,
    plot_mpet_single_panel, interactive_threshold_toggle,
    plot_mpet_bioclimate_diagram,
    plot_acclimatization_summary,
    plot_benefited_area, plot_benefited_population,
    plot_benefited_population_maps,
    plot_prolonged_exposure_maps,
)

# Clean display
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*crs.*')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('All libraries loaded successfully.')

To account for variability in human thermoregulatory responses, the reproduced study computes mPET for **four population groups**: male adults, female adults, male seniors, and female seniors. Each run of this notebook focuses on one of these four groups. **Three cool-roof scenarios** are applied: white or cool-colored paint, reflective coating, and super-cool material. Of these, super-cool material is used as the default material option, so the heat-stress response results compare on the same basis in Section 2. Similarly, the heat-vulnerable **West Athens** is used as the default focus regional unit, and the default focus day is set to **3 August** (03/08), within the simulated nine-day heat wave, during which an observed maximum air temperature of 43.9 °C was recorded. Together with Central Athens and Piraeus, West Athens is one of three regional units that make up the intervention's broader influence zone surrounding the Elaionas zone.

All four settings, population group, cool-roof material, focus region, and focus day, are configurable in the cell below.


In [ ]:
# --- Population group setting ----------------------------------------------
GROUP_KEY = 'Female_Adults'   # [Group-specific; options: Female_Adults, Male_Adults, Female_Seniors, Male_Seniors]
assert GROUP_KEY in POPULATION_GROUPS, (
    f'Invalid group: {GROUP_KEY}. Choose from {list(POPULATION_GROUPS)}'
)
GROUP = POPULATION_GROUPS[GROUP_KEY]

# --- Defaults (changing these affects notebook outputs) --------------------
MATERIAL = 'supercool'          # [Section 2; options: white_paint, reflective, supercool]
FOCUS_REGION = 'West_Athens' # [Section 2; options: Central_Athens, West_Athens, Piraeus]
FOCUS_DAY = (8, 3)              # [Section 2; range: (7, 28) to (8, 5)]

# --- Paths (do not modify) ------------------------------------------------
DATA_DIR = Path('..') / 'data' 
WRF_DIR = DATA_DIR / 'wrf_400m_output'
RAYMAN_DIR = DATA_DIR / 'rayman_output'
POP_DIR = DATA_DIR / 'population'
MASK_DIR = DATA_DIR / 'region_masks'
OUT_DIR = Path('..') / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Shapefile paths (do not modify) --------------------------------------
SHP_AUA = MASK_DIR / 'aua_regional_units.shp'
SHP_ELAIONAS = MASK_DIR / 'elaionas_intervention_zone.shp'
SHP_ATTICA = MASK_DIR / 'attica_boundary.shp'

# --- Load acclimatized thresholds (do not modify) -------------------------
THR_DF = load_strong_heat_stress_thresholds(DATA_DIR)

# --- Population totals (do not modify) ------------------------------------
POP_TOTALS_DF = pd.read_csv(POP_DIR / 'AUA_Regional_Units_PopTotals_2021.csv')

print(f'Group: {GROUP["label"]}')
print(f'Material: {SCENARIOS[MATERIAL]["label"]}')
print(f'Focus region: {REGIONS[FOCUS_REGION]["label"]}')
print(f'Focus day: {FOCUS_DAY[1]:02d}/{FOCUS_DAY[0]:02d}')
print(f'Data directory: {DATA_DIR.resolve()}')

---

## Section 1: Atmospheric cooling

The Elaionas intervention zone is classified mainly as **Local Climate Zone (LCZ) 8**, where the sea breeze, especially under heat-wave conditions, penetrates the low-rise fabric and advects cooler air toward **West Athens**. The three cool-roof scenarios are therefore compared against the base case under two wind patterns, namely the **Etesians** (persistent northerly flow) phase (HW0) and the **sea-breeze** phase (HW1)

### 1.1 Near-surface conditions over the influence zone

The **base case** represents conditions before any cool-roof scenario is applied and serves as the reference for the scenario comparisons. Area-mean time series of **2 m air temperature (T2)** are built in the cell below over the three regional units across the nine-day heat wave, with the Etesians and sea-breeze phases shaded.

In [ ]:
# Load precomputed area-mean base-case T2 for the three regional units
region_series = load_base_case_area_means(data_dir=DATA_DIR)

# Identify contiguous HW phase spans for background shading
hw0_spans, hw1_spans = phase_spans(times=region_series['Central_Athens']['time'])

# Single-panel time series: three regions overlaid, HW phases hatched
plot_base_t2_timeseries(
    region_series=region_series,
    hw0_spans=hw0_spans,
    hw1_spans=hw1_spans,
    out_dir=OUT_DIR,
)

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 1.1: Regional and Diurnal Patterns in the Base Case

**Goal**: Interpret the base-case time-series figure before any cooling intervention is introduced.

1. Which regional unit looks consistently warmest across the whole time series, day and night?
2. Where do the largest regional differences show up, under the Etesians phase or the sea-breeze phase?
3. Do nighttime temperatures drop enough to offer nocturnal thermal relief?

</div>

---

### 1.2 Air-temperature reductions by scenario and wind phase

The magnitude and reach of the cooling depend on the cool-roof scenario and the prevailing flow. The daytime air-temperature difference between each of the **three scenarios** and the base case is mapped below across both wind phases in a single panel, with the **Elaionas** area outlined and 10 m wind vectors overlaid.

In [ ]:
# 2x3 panel maps of daytime mean delta-T2 with 10 m wind vectors
plot_diff_panel_maps(
    wrf_dir=WRF_DIR,
    shp_aua=SHP_AUA,
    shp_attica=SHP_ATTICA,
    shp_elaionas=SHP_ELAIONAS,
    out_dir=OUT_DIR,
)

### 1.3 Diurnal evolution of cooling

Cooling magnitude changes through the day. The **hourly** air-temperature reduction for each cool-roof scenario is traced in the cell that follows, separated by regional unit and wind phase.

In [ ]:
# Hourly cooling by scenario, separated by regional unit and HW phase
plot_cooling_heatmap(
    wrf_dir=WRF_DIR,
    out_dir=OUT_DIR,
)

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 1.2: Interpreting the cooling maps and diurnal evolution

**Goal**: Identify the key spatial and temporal patterns of the cooling intervention.

1. How does the spatial footprint of cooling shift between the Etesians phase (HW0) and the sea-breeze phase (HW1)?
2. Which regional unit shows the strongest cooling, and during which hours does it peak?
3. How do the three cool-roof scenarios rank in cooling performance?

</div>

---

## Section 2: Heat-stress response (mPET)

mPET integrates the four atmospheric drivers behind heat stress, namely air temperature, humidity, wind speed, and mean radiant temperature, with a person's thermo-physiological characteristics, namely sex, age, body size, metabolic activity, and clothing. The same atmospheric conditions therefore produce different heat-stress levels for female adults, male adults, female seniors, and male seniors. The mPET values used in this notebook were computed with the **RayMan Pro** model for the four population groups at the **grid points showing air-temperature reductions**.

### 2.1 mPET thermal bioclimate diagram

The **hourly frequency** of the standard mPET thermal stress categories is derived below for the focus region (West Athens by default) and cool-roof scenario (super-cool material by default) across the heat wave. The diagrams for all four populations appear side by side, making it possible to assess the timing and persistence of heat stress through the day and night, and to compare the heat-stress burden across groups.


In [ ]:
# 2x2 bioclimate diagram: hourly mPET stress categories, all four groups
plot_mpet_bioclimate_diagram(
    rayman_dir=RAYMAN_DIR,
    region=FOCUS_REGION,
    material=MATERIAL,
    out_dir=OUT_DIR,
)


---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.1: Severity and persistence of heat stress

**Goal**: Interpret the mPET thermal bioclimate diagrams across the four population groups.

1. During which hours of the day does strong or extreme heat stress dominate the distribution, and which atmospheric factors are likely driving this pattern?
2. Is the absence of nocturnal thermal relief identified in Task 1.1 reflected in the mPET categories overnight?

</div>

---

### 2.2 Acclimatization-based thresholds

In practice, the mPET thresholds used to define the various heat-stress levels are not fixed. They vary for every population and place due to **short-term acclimatization**, reflecting thermophysiological differences across groups and local climate differences. To account for this, fixed heat-stress thresholds are adjusted with a Gaussian filter that represents short-term (30-day) acclimatization, giving population- and location-specific thresholds for the focus region during the heat wave. 

The interactive toggle below illustrates this adaptation for the strong-heat-stress threshold, showing the heat-stress distribution at its fixed and at its acclimatized value for the cool-roof scenario (super-cool material by default), the configured group (Female Adults by default), and the focus day (3 August by default). The summary bar chart that follows compares the fixed and acclimatized exceedance percentages for **strong and extreme heat stress** across all four population groups.


In [ ]:
# Binary toggle: compare fixed vs. acclimatized threshold on the focus day
interactive_threshold_toggle(
    rayman_dir=RAYMAN_DIR,
    region=FOCUS_REGION,
    group_key=GROUP_KEY,
    material=MATERIAL,
    thr_df=THR_DF,
    focus_day=FOCUS_DAY,
)


In [ ]:
# Fixed vs. per-day acclimatized exceedance across all four groups
plot_acclimatization_summary(
    rayman_dir=RAYMAN_DIR,
    region=FOCUS_REGION,
    thr_df=THR_DF,
    material=MATERIAL,
    out_dir=OUT_DIR,
)


---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.2: Acclimatization and its effect on exceedance

**Goal**: Interpret how shifting from the fixed to the acclimatized threshold changes each population group's heat-stress exceedance.

1. Which population group shows the highest exceedance of strong and extreme heat stress, and which thermo-physiological drivers may explain this ranking?
2. What would be misrepresented about a population's heat-stress exposure if acclimatization were ignored and only the fixed threshold were applied?

</div>

---

### 2.3 Prolonged heat-stress exposure

The duration of strong heat stress exposure is a key determinant of its health impact. For AUA specifically, daily exposure to strong heat stress above six hours is associated with up to a 56% increase in heat-related mortality risk. For the configured population group (Female Adults by default) and cool-roof scenario (super-cool material by default), the number of days of each wind phase that each cooling grid point exceeds the acclimatized strong-heat-stress threshold for more than six hours is mapped below across all three regional units.


In [ ]:
# Side-by-side folium maps: days with >6h strong heat stress per grid point
plot_prolonged_exposure_maps(
    rayman_dir=RAYMAN_DIR,
    thr_df=THR_DF,
    group_key=GROUP_KEY,
    material=MATERIAL,
    mask_dir=MASK_DIR,
    out_dir=OUT_DIR,
)


---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.3: Interpreting prolonged exposure for the configured group

**Goal**: Interpret the spatial pattern of prolonged heat-stress exposure for the configured population group.

1. Which regional unit shows the highest prevalence of prolonged exposure under HW1?
2. How does the spatial pattern differ between HW0 and HW1?

</div>


---

## Section 3: Health-related benefits

A health-related benefit is defined as a reduction in daily maximum mPET at cooling grid points identified in Section 2.3 as exceeding six hours of strong heat stress per day. A lower peak mPET under a given cool-roof scenario counts as a benefit even if the six-hour threshold continues to be exceeded, since a lower value corresponds to lower mortality risk. Combining the daily benefited grid points with gridded population data by sex and age yields the average **benefited population** by group, extending the assessment to the dimensions of just resilience.

### 3.1 Benefited area

A **benefited area** is defined as the percentage of a regional unit's grid points classified as benefited on a given day. For the focus group (Female Adults by default), this is computed below for each day of each wind phase and cool-roof scenario. The results are presented as grouped bar charts, one panel per regional unit and wind phase, with bars grouped by scenario per day.

In [ ]:
# 2x3 panel of benefited-area % by scenario, region, and HW phase
plot_benefited_area(
    rayman_dir=RAYMAN_DIR,
    thr_df=THR_DF,
    group_key=GROUP_KEY,
    out_dir=OUT_DIR,
)

### 3.2 Benefited population

Gridded population counts convert the benefited area computed for each day in Section 3.1 into an absolute number of people. These counts are averaged across the days under HW0 (Etesians) and HW1 (sea-breeze) conditions. For the focus group (Female Adults by default), this is computed below, with both the absolute number and the percentage of the group's total population provided. An interactive map, also produced below for the cool-roof scenario (super-cool material by default), shows these grid points sized by the population they represent, making visible where this population is concentrated within the intervention's broader influence zone.

In [ ]:
# Mean daily benefited population per region and scenario
plot_benefited_population(
    rayman_dir=RAYMAN_DIR,
    thr_df=THR_DF,
    group_key=GROUP_KEY,
    pop_dir=POP_DIR,
    out_dir=OUT_DIR,
)


In [ ]:
# Side-by-side folium maps: benefited cells sized by population
plot_benefited_population_maps(
    rayman_dir=RAYMAN_DIR,
    thr_df=THR_DF,
    group_key=GROUP_KEY,
    material=MATERIAL,
    pop_dir=POP_DIR,
    mask_dir=MASK_DIR,
    out_dir=OUT_DIR,
)

---

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 3.1: Just-resilience synthesis

**Goal**: Assess what the benefited-area and benefited-population outcomes imply for distributional, recognitional, and procedural justice for the configured population group.

1. **Distributional justice**: Which regional unit receives the greatest benefit under HW1 for the configured population group, and how does this compare with where heat exposure and vulnerability are greatest?
2. **Recognitional justice**: How do the population-specific heat-stress thresholds and benefited-population results differ from an estimate based on a single undifferentiated population?
3. **Procedural justice**: Which affected communities and public authorities should participate in shaping the cool-roof intervention, and what local knowledge or priorities should inform its practical deployment?

</div>

---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>

*A work conducted in collaboration with the National Observatory of Athens and funded by Horizon Europe [ClimateAdapt4EOSC](https://climate-adapt4eosc.eu/) (Grant No. 101188248)*
